In [20]:
import pandas as pd
from pathlib import Path
from matplotlib import pyplot as plt
import os



from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

# Tree Visualisation
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [69]:
def create_inference_column_per_samples(dataset_name):


        # Create directory
        # dir_path_healthy = f"{dataset_name}/random_rows/{int(p*100)}/healthy"
        # dir_path_unhealthy = f"{dataset_name}/random_rows/{int(p*100)}/unhealthy"

    df_main_meta = pd.read_csv(f"../processed_data/{dataset_name}/metadata.tsv", sep='\t')
        # os.makedirs(dir_path_healthy, exist_ok=True)
        # os.makedirs(dir_path_unhealthy, exist_ok=True)

    if dataset_name == "iHMP_IBDMDB_2019":
        meta_sub = df_main_meta[['Sample', 'Study.Group']].copy()
        meta_sub['is_healthy'] = (meta_sub['Study.Group'] == 'nonIBD').astype(int)
        # 3. Finalize the meta_sub selection
        
    elif dataset_name == "WANG_ESRD_2020":
        meta_sub = df_main_meta[['Sample', 'Study.Group']].copy()

        meta_sub['is_healthy'] = (meta_sub['Study.Group'] == 'Control').astype(int)
    
    elif dataset_name == "YACHIDA_CRC_2019":
        meta_sub = df_main_meta[['Sample', 'Study.Group']].copy()

        meta_sub['is_healthy'] = (meta_sub['Study.Group'] == 'Healthy').astype(int)

    meta_sub = meta_sub[['Sample', 'is_healthy']]
    combined_df = pd.merge(df_main_meta, meta_sub, on='Sample', how='left')

    sample_is_healthy_df = combined_df[['Sample', 'is_healthy']].copy()

    sizes = ['30', '60', '90']
    num_iterations = 100
    rows_or_columns = ['random_rows', 'random_columns']
    samples_information = []

    for index, row in sample_is_healthy_df.iterrows():
        sample_id = row['Sample']
        status = row['is_healthy']

        healthy_or_unhealthy = 'healthy' if status == 1 else 'unhealthy'
        paths = []
        values_inference = []
        for p in sizes:
            for random_type in rows_or_columns:
                for i in range(1, num_iterations + 1):
                    filename = f"sample_{p}_run_{i}.tsv"
                    
                    path = f"{dataset_name}/{random_type}/{p}/{healthy_or_unhealthy}/{filename}"

                    if not os.path.isfile(path):
                        continue
                    rows_or_columns_2 = 'rows' if random_type == 'random_rows' else 'columns'
                    path_results = f"results/df_{healthy_or_unhealthy}_30_60_90_{dataset_name}_{rows_or_columns_2}.tsv"

                    sample_df = pd.read_csv(path, sep='\t')
                    if sample_id in sample_df['Sample'].values:
                        results_df = pd.read_csv(path_results, sep='\t')
                        id_row = i - 1
                        if p == '60':
                            id_row += 100
                        elif p == '90':
                            id_row += 200

                        row_inference = results_df.iloc[id_row]['Rho']
                        values_inference.append(row_inference)
                        paths.append(path)

        
        samples_information.append({
            "Sample": sample_id,
            "average_rho": sum(values_inference) / len(values_inference),
            'number_of_rho_values': len(values_inference),
            'number_of_rho_paths': len(paths),
            "paths": paths,
            "rho_values": values_inference
        })

    results = pd.DataFrame(samples_information)
    path_to_save = f"results_inference/{dataset_name}_inference.tsv"
    results.to_csv(path_to_save, sep="\t", index=True)

In [70]:
create_inference_column_per_samples("WANG_ESRD_2020")
create_inference_column_per_samples("YACHIDA_CRC_2019")
create_inference_column_per_samples("iHMP_IBDMDB_2019")

In [59]:
df_meta_WANG_ESRD_2020 = pd.read_csv('../processed_data/WANG_ESRD_2020/metadata.tsv', sep='\t')
df_meta_YACHIDA_CRC_2019 = pd.read_csv('../processed_data/YACHIDA_CRC_2019/metadata.tsv', sep='\t')
df_meta_iHMP_IBDMDB_2019 = pd.read_csv('../processed_data/iHMP_IBDMDB_2019/metadata.tsv', sep='\t')


In [60]:
df_genera_WANG_ESRD_2020 = pd.read_csv('../processed_data/WANG_ESRD_2020/genera.tsv', sep='\t')
df_genera_YACHIDA_CRC_2019 = pd.read_csv('../processed_data/YACHIDA_CRC_2019/genera.tsv', sep='\t')
df_genera_iHMP_IBDMDB_2019 = pd.read_csv('../processed_data/iHMP_IBDMDB_2019/genera.tsv', sep='\t')


In [61]:
df_inference_WANG_ESRD_2020 = pd.read_csv('./results_inference/WANG_ESRD_2020_inference.tsv', sep="\t",)
df_inference_YACHIDA_CRC_2019 = pd.read_csv('./results_inference/YACHIDA_CRC_2019_inference.tsv', sep="\t",)
df_inference_iHMP_IBDMDB_2019 = pd.read_csv('./results_inference/iHMP_IBDMDB_2019_inference.tsv', sep="\t",)

# WANG_ESRD_2020_inference

In [73]:
df_meta_WANG_ESRD_2020.columns

Index(['Dataset', 'Sample', 'Subject', 'Study.Group', 'Age', 'Age.Units',
       'Gender', 'BMI', 'DOI', 'Publication.Name', 'Source Name', 'Creatinine',
       'Dialysis.Frequency', 'Urea', 'eGFR'],
      dtype='str')

In [74]:
df_inference_WANG_ESRD_2020.columns

Index(['Unnamed: 0', 'Sample', 'average_rho', 'number_of_rho_values',
       'number_of_rho_paths', 'paths', 'rho_values'],
      dtype='str')

In [5]:

df_meta_WANG_ESRD_2020['is_healthy'] = (df_meta_WANG_ESRD_2020['Study.Group'] == 'Control').astype(int)

In [6]:
df_meta_WANG_ESRD_2020 = df_meta_WANG_ESRD_2020.drop(columns=[
	'Dataset', 
	'Subject', 
	'Study.Group',
	'Age.Units', 
	'DOI', # link to the original study
	'Publication.Name', 
	'Source Name', 
	'Urea', # In ESRD: High levels of urea in the blood (uremia) can cause symptoms like fatigue and nausea.
	'Dialysis.Frequency',
	'eGFR', # The Scale: It tells you how many milliliters of blood your kidneys filter per minute.
	'Creatinine' # If kidneys are failing, creatinine levels in the blood rise
])

df_inference_WANG_ESRD_2020 = df_inference_WANG_ESRD_2020.drop(columns=[
	'Unnamed: 0',
	'number_of_rho_values',
    'number_of_rho_paths', 
	'paths', 
	'rho_values'
])

In [7]:
combined_df_inference = pd.merge(df_meta_WANG_ESRD_2020, df_inference_WANG_ESRD_2020, on='Sample', how='left')

In [94]:
combined_df_inference

,Sample,Age,Gender,BMI,is_healthy,average_rho
0,CON-001,62,Female,19.570000,1,0.868524
1,CON-002,61,Female,20.820000,1,0.877513
2,CON-003,60,Female,23.830000,1,0.875212
3,CON-004,46,Female,NaN,1,0.872796
4,CON-005,45,Female,21.190000,1,0.873714
...,...,...,...,...,...,...
282,KD-219,34,Female,22.768834,0,0.945538
283,KD-220,37,Male,28.408163,0,0.946350
284,KD-221,35,Male,22.857143,0,0.946257
285,KD-222,34,Female,20.474990,0,0.947076


In [8]:
combined_df_inference_and_genera = pd.merge(combined_df_inference, df_genera_WANG_ESRD_2020, on='Sample', how='left')

In [9]:
combined_df_inference_and_genera.set_index('Sample', inplace=True)	

In [10]:
# One-hot encode Gender column (convert to 1 or 0)
combined_df_inference_and_genera['Gender'] = (combined_df_inference_and_genera['Gender'] == 'Male').astype(int)

# Fill NaN values in BMI with 0
combined_df_inference_and_genera['BMI'] = combined_df_inference_and_genera['BMI'].fillna(0)

In [11]:
combined_df_inference_and_genera

,Age,Gender,BMI,is_healthy,average_rho,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Peptostreptococcales;f__Acidaminobacteraceae;g__Fusibacter_A,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__CAG-272;g__Flemingibacterium,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__F0428,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Lachnoanaerobaculum,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactobacillales;f__Carnobacteriaceae;g__Carnobacterium_A,...,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodobacterales;f__Rhodobacteraceae;g__Cribrihabitans,d__Bacteria;p__Patescibacteria;c__Microgenomatia;o__Curtissbacterales;f__GWA2-41-24;g__MFBO01,d__Bacteria;p__Patescibacteria;c__Paceibacteria;o__UBA6257;f__2-01-FULL-56-20;g__VMFW01,d__Bacteria;p__Actinobacteriota;c__Thermoleophilia;o__Gaiellales;f__Gaiellaceae;g__JACCYG01,d__Bacteria;p__Patescibacteria;c__ABY1;o__SG8-24;f__UBA9934;g__JAHISC01,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rickettsiales;f__Rickettsiaceae;g__CAJBEU01,d__Bacteria;p__Patescibacteria;c__ABY1;o__SG8-24;f__GWF2-40-263;g__JACPHX01,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhizobiales;f__Xanthobacteraceae;g__GCF-000702305,d__Bacteria;p__Firmicutes;c__Bacilli;o__Izemoplasmatales;f__UBA5603;g__M55B122,d__Bacteria;p__Patescibacteria;c__ABY1;o__SBBC01;f__PEXW01;g__CAIPSZ01
Sample,,,,,,,,,,,,,,,,,,,,,
CON-001,62,0,19.570000,1,0.868524,0.000004,0.000004,0.000065,0.000128,1.306828e-05,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
CON-002,61,0,20.820000,1,0.877513,0.000002,0.000004,0.000005,0.000014,6.546411e-07,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
CON-003,60,0,23.830000,1,0.875212,0.000001,0.000002,0.000004,0.000119,1.390039e-06,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
CON-004,46,0,0.000000,1,0.872796,0.000001,0.000179,0.000006,0.000042,1.282748e-06,...,0.0,0.0,0.0,0.0,3.043809e-07,2.391564e-07,2.174149e-07,7.392107e-07,0.000002,3.261224e-07
CON-005,45,0,21.190000,1,0.873714,0.000001,0.000011,0.000007,0.000025,8.170867e-07,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
KD-219,34,0,22.768834,0,0.945538,0.000004,0.000172,0.000040,0.000071,1.707281e-06,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
KD-220,37,1,28.408163,0,0.946350,0.000002,0.000001,0.000008,0.000025,6.103054e-07,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
KD-221,35,1,22.857143,0,0.946257,0.000001,0.000007,0.000026,0.000073,4.281503e-06,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00


In [12]:
X = combined_df_inference_and_genera.drop('is_healthy', axis=1)
y = combined_df_inference_and_genera['is_healthy']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [14]:
y_pred = rf.predict(X_test)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8275862068965517


In [18]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print(importances)

average_rho                                                                                                0.034318
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__Oscillospiraceae;g__Lawsonibacter          0.009316
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__UBA9414                  0.006663
d__Bacteria;p__Firmicutes;c__Bacilli;o__Erysipelotrichales;f__Erysipelotrichaceae;g__Merdibacter           0.006080
d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactobacillales;f__Enterococcaceae;g__Enterococcus_C               0.005873
                                                                                                             ...   
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;f__Rhodocyclaceae;g__JAGTRP01      0.000000
d__Bacteria;p__Firmicutes;c__Bacilli;o__Bacillales_D;f__Amphibacillaceae;g__Lentibacillus_B                0.000000
d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Burkholderiales;

In [19]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance on the test/validation set
result = permutation_importance(
    rf, X, y, n_repeats=10, random_state=42, n_jobs=-1
)

# Organize into a Series
perm_importances = pd.Series(result.importances_mean, index=X.columns)
perm_importances = perm_importances.sort_values(ascending=False)

print(perm_importances)

average_rho                                                                                                   8.362369e-03
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__UBA3855                    5.923345e-03
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__UBA9414                     5.226481e-03
d__Bacteria;p__Actinobacteriota;c__Actinomycetia;o__Mycobacteriales;f__Micromonosporaceae;g__Phytohabitans    3.484321e-03
d__Bacteria;p__Bacteroidota;c__Rhodothermia;o__Balneolales;f__Balneolaceae;g__Gracilimonas                    3.135889e-03
                                                                                                                  ...     
d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhizobiales;f__Rhizobiaceae;g__Nitratireductor_D      0.000000e+00
d__Bacteria;p__Patescibacteria;c__ABY1;o__SBBC01;f__PEXW01;g__CAIPSZ01                                        0.000000e+00
d__Bacteria;p__B

In [21]:



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Initialize and train the Support Vector Classifier
# 'C' is the regularization parameter
model = SVC(kernel='linear', C=1.0)
model.fit(X_train, y_train)

# 5. Predict and Evaluate
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.90
              precision    recall  f1-score   support

           0       0.90      0.98      0.93        44
           1       0.90      0.64      0.75        14

    accuracy                           0.90        58
   macro avg       0.90      0.81      0.84        58
weighted avg       0.90      0.90      0.89        58



# YACHIDA_CRC_2019

In [24]:
df_meta_YACHIDA_CRC_2019.columns

Index(['Dataset', 'Sample', 'Subject', 'Study.Group', 'Age', 'Age.Units',
       'Gender', 'BMI', 'DOI', 'Publication.Name', 'Stage', 'Brinkman Index',
       'Alcohol', 'Tumor location', 'Shared.w.ERAWIJANTARI_2020'],
      dtype='str')

In [25]:
df_inference_YACHIDA_CRC_2019.columns

Index(['Unnamed: 0', 'Sample', 'average_rho', 'number_of_rho_values',
       'number_of_rho_paths', 'paths', 'rho_values'],
      dtype='str')

In [26]:

df_meta_YACHIDA_CRC_2019['is_healthy'] = (df_meta_YACHIDA_CRC_2019['Study.Group'] == 'Healthy').astype(int)

In [30]:
df_meta_YACHIDA_CRC_2019 = df_meta_YACHIDA_CRC_2019.drop(columns=[
	'Dataset', 
	'Subject', 
	'Study.Group',
	'Age.Units', 
	'DOI', # link to the original study
	'Publication.Name', 
	'Stage',
	'Brinkman Index',
	'Alcohol',
	'Tumor location',
	'Shared.w.ERAWIJANTARI_2020',
])

df_inference_YACHIDA_CRC_2019 = df_inference_YACHIDA_CRC_2019.drop(columns=[
	'Unnamed: 0',
	'number_of_rho_values',
    'number_of_rho_paths', 
	'paths', 
	'rho_values'
])

In [31]:
combined_df_inference_YACHIDA_CRC_2019 = pd.merge(df_meta_YACHIDA_CRC_2019, df_inference_YACHIDA_CRC_2019, on='Sample', how='left')

In [33]:
combined_df_inference_YACHIDA_CRC_2019

,Sample,Age,Gender,BMI,is_healthy,average_rho
0,10021,57,Male,26.880952,0,0.941019
1,10023,65,Male,26.562500,1,0.952693
2,10025,40,Male,25.000000,1,0.948712
3,10029,67,Female,20.173253,1,0.947734
4,10031,77,Male,24.464602,1,0.948226
...,...,...,...,...,...,...
342,10838,75,Female,21.170218,1,0.948313
343,10839,58,Male,19.605192,0,0.937660
344,10847,73,Female,20.545694,0,0.936914
345,10850,51,Male,28.384802,1,0.947940


In [34]:
combined_df_inference_and_genera_YACHIDA_CRC_2019 = pd.merge(combined_df_inference_YACHIDA_CRC_2019, df_genera_YACHIDA_CRC_2019, on='Sample', how='left')

In [35]:
# One-hot encode Gender column (convert to 1 or 0)
combined_df_inference_and_genera_YACHIDA_CRC_2019['Gender'] = (combined_df_inference_and_genera_YACHIDA_CRC_2019['Gender'] == 'Male').astype(int)

# Fill NaN values in BMI with 0
combined_df_inference_and_genera_YACHIDA_CRC_2019['BMI'] = combined_df_inference_and_genera_YACHIDA_CRC_2019['BMI'].fillna(0)

In [36]:
combined_df_inference_and_genera_YACHIDA_CRC_2019

,Sample,Age,Gender,BMI,is_healthy,average_rho,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Monoglobales;f__Firm-18;g__UBA1775,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Peptostreptococcales;f__Acidaminobacteraceae;g__Fusibacter_A,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodobacterales;f__Rhodobacteraceae;g__Oceaniglobus,d__Bacteria;p__Myxococcota;c__Polyangia;o__Polyangiales;f__SG8-38;g__JAAYBZ01,...,d__Bacteria;p__Patescibacteria;c__Kazan-3B-28;o__Kazan-3B-28;f__UBA10110;g__JAHITC01,d__Bacteria;p__Patescibacteria;c__ABY1;o__SG8-24;f__2-12-FULL-60-25;g__RHKK01,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rickettsiales;f__UBA3002;g__UBA6178,d__Bacteria;p__Firmicutes;c__Bacilli;o__Mycoplasmatales;f__MT37;g__Spiroplasma_C,d__Bacteria;p__Desulfobacterota_E;c__MBNT15;o__J040;f__J040;g__WVXK01,d__Bacteria;p__Patescibacteria;c__Paceibacteria;o__2-02-FULL-40-12;f__IGHO2-01-FULL-4-A;g__VMGJ01,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodobacterales;f__Rhodobacteraceae;g__JABSSB01,d__Bacteria;p__Patescibacteria;c__Microgenomatia;o__Curtissbacterales;f__GWA2-41-24;g__JACQIV01,d__Bacteria;p__Chloroflexota;c__Chloroflexia;o__Thermobaculales;f__Thermobaculaceae;g__LC3-1,d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Pseudomonadales;f__HTCC2089;g__UBA7451
0,10021,57,1,26.880952,0,0.941019,0.000000e+00,1.118365e-06,0.000000e+00,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
1,10023,65,1,26.562500,1,0.952693,0.000000e+00,3.728608e-06,0.000000e+00,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
2,10025,40,1,25.000000,1,0.948712,0.000000e+00,1.393738e-06,9.466900e-07,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
3,10029,67,0,20.173253,1,0.947734,0.000000e+00,1.612813e-06,4.543136e-07,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
4,10031,77,1,24.464602,1,0.948226,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,10838,75,0,21.170218,1,0.948313,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
343,10839,58,1,19.605192,0,0.937660,0.000000e+00,3.891475e-07,0.000000e+00,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
344,10847,73,0,20.545694,0,0.936914,0.000000e+00,1.495768e-06,5.711114e-07,3.535452e-07,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0
345,10850,51,1,28.384802,1,0.947940,0.000000e+00,6.333586e-07,0.000000e+00,0.000000e+00,...,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0.0


In [42]:
X = combined_df_inference_and_genera_YACHIDA_CRC_2019.drop('is_healthy', axis=1)
y = combined_df_inference_and_genera_YACHIDA_CRC_2019['is_healthy']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [47]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [48]:
y_pred = rf.predict(X_test)

In [49]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.6714285714285714


In [50]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print(importances)

average_rho                                                                                                             0.045878
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__P3;g__Pullibacteroides                                   0.003859
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Marinifilaceae;g__Odoribacter                            0.003294
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Lachnospira                           0.003220
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Christensenellales;f__Christensenellaceae;g__Christensenella               0.002967
                                                                                                                          ...   
d__Bacteria;p__Acidobacteriota;c__Blastocatellia;o__RBC074;f__RBC074;g__JADJLO01                                        0.000000
d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Flavobacteriaceae;g__SMXJ01    

In [51]:



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Initialize and train the Support Vector Classifier
# 'C' is the regularization parameter
model = SVC(kernel='linear', C=1.0)
model.fit(X_train, y_train)

# 5. Predict and Evaluate
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.64
              precision    recall  f1-score   support

           0       0.67      0.84      0.75        44
           1       0.53      0.31      0.39        26

    accuracy                           0.64        70
   macro avg       0.60      0.57      0.57        70
weighted avg       0.62      0.64      0.61        70



# iHMP_IBDMDB_2019

In [62]:
df_meta_iHMP_IBDMDB_2019

,Dataset,Sample,Subject,Study.Group,Gender,DOI,Publication.Name,consent_age,Age.Units,site_sub_coll,...,visit_num,site_name,Age at diagnosis,Antibiotics,race,fecalcal,BMI_at_baseline,Height_at_baseline,Weight_at_baseline,smoking status
0,iHMP_IBDMDB_2019,CSM5MCVN,C3002,CD,Female,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,76.0,Years,C3002C9,...,13,Cedars-Sinai,47.0,No,White,15.97901,NaN,NaN,NaN,NaN
1,iHMP_IBDMDB_2019,CSM5MCWE,C3009,CD,Male,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,56.0,Years,C3009C5,...,8,Cedars-Sinai,44.0,No,White,20.64059,NaN,NaN,NaN,NaN
2,iHMP_IBDMDB_2019,CSM5MCX3,C3006,UC,Male,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,32.0,Years,C3006C9,...,13,Cedars-Sinai,24.0,No,White,12.69817,20.1,180.0,65.0,Never smoked
3,iHMP_IBDMDB_2019,CSM5MCXL,C3004,UC,Female,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,47.0,Years,C3004C9,...,13,Cedars-Sinai,33.0,No,White,14.82410,NaN,NaN,NaN,NaN
4,iHMP_IBDMDB_2019,CSM5MCY8,C3005,UC,Female,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,76.0,Years,C3005C9,...,13,Cedars-Sinai,58.0,No,White,229.04730,30.9,165.0,84.0,Former smoker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377,iHMP_IBDMDB_2019,PSMA267P,P6035,UC,Male,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,16.0,Years,P6035C11,...,15,MGH Pediatrics,16.0,No,White,14.57958,NaN,NaN,NaN,NaN
378,iHMP_IBDMDB_2019,PSMA269W,P6038,UC,Female,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,16.0,Years,P6038C10,...,14,MGH Pediatrics,8.0,No,White,NaN,NaN,NaN,NaN,NaN
379,iHMP_IBDMDB_2019,PSMA26A3,P6038,UC,Female,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,16.0,Years,P6038C13,...,18,MGH Pediatrics,8.0,No,White,NaN,NaN,NaN,NaN,NaN
380,iHMP_IBDMDB_2019,PSMB4MBK,P6035,UC,Male,10.1038/s41586-019-1237-9,Multi-omics of the gut microbial ecosystem in ...,16.0,Years,P6035C16,...,21,MGH Pediatrics,16.0,No,White,127.56480,NaN,NaN,NaN,NaN


In [63]:
df_meta_iHMP_IBDMDB_2019.isnull().sum()

Dataset                 0
Sample                  0
Subject                 0
Study.Group             0
Gender                  0
DOI                     0
Publication.Name        0
consent_age             4
Age.Units               0
site_sub_coll           0
ProjectSpecificID       0
week_num                0
date_of_receipt         0
interval_days           0
visit_num               0
site_name               0
Age at diagnosis      108
Antibiotics             0
race                    0
fecalcal              161
BMI_at_baseline        87
Height_at_baseline     87
Weight_at_baseline     87
smoking status        215
dtype: int64

In [64]:
df_meta_iHMP_IBDMDB_2019['race'].value_counts()

race
White                               343
More than one race                   17
Black or African American            14
Other                                 5
American Indian or Alaska Native      3
Name: count, dtype: int64

In [65]:
df_meta_iHMP_IBDMDB_2019.columns

Index(['Dataset', 'Sample', 'Subject', 'Study.Group', 'Gender', 'DOI',
       'Publication.Name', 'consent_age', 'Age.Units', 'site_sub_coll',
       'ProjectSpecificID', 'week_num', 'date_of_receipt', 'interval_days',
       'visit_num', 'site_name', 'Age at diagnosis', 'Antibiotics', 'race',
       'fecalcal', 'BMI_at_baseline', 'Height_at_baseline',
       'Weight_at_baseline', 'smoking status'],
      dtype='str')

In [66]:
df_inference_iHMP_IBDMDB_2019.columns

Index(['Unnamed: 0', 'Sample', 'average_rho', 'number_of_rho_values',
       'number_of_rho_paths', 'paths', 'rho_values'],
      dtype='str')

In [67]:
df_meta_iHMP_IBDMDB_2019['is_healthy'] = (df_meta_iHMP_IBDMDB_2019['Study.Group'] == 'nonIBD').astype(int)

In [68]:
df_meta_iHMP_IBDMDB_2019 = df_meta_iHMP_IBDMDB_2019.drop(columns=[
	'Dataset', 
	'Subject', 
	'Study.Group',
	'Age.Units', 
	'ProjectSpecificID',
	'consent_age',
	'Age.Units',
	'site_sub_coll',
	'ProjectSpecificID', 
	'week_num', 
	'date_of_receipt', 
	'interval_days',
	'visit_num', 
	'site_name', 
	'Age at diagnosis', 
	'Antibiotics',
	'race',
	'fecalcal',
	'smoking status',
	'DOI', # link to the original study
	'Publication.Name', 
])

df_inference_iHMP_IBDMDB_2019 = df_inference_iHMP_IBDMDB_2019.drop(columns=[
	'Unnamed: 0',
	'number_of_rho_values',
    'number_of_rho_paths', 
	'paths', 
	'rho_values'
])

In [69]:
# One-hot encode Gender column (convert to 1 or 0)
df_meta_iHMP_IBDMDB_2019['Gender'] = (df_meta_iHMP_IBDMDB_2019['Gender'] == 'Male').astype(int)

# Fill NaN values in BMI with 0
df_meta_iHMP_IBDMDB_2019['Height_at_baseline'] = df_meta_iHMP_IBDMDB_2019['Height_at_baseline'].fillna(0)
df_meta_iHMP_IBDMDB_2019['Weight_at_baseline'] = df_meta_iHMP_IBDMDB_2019['Weight_at_baseline'].fillna(0)
df_meta_iHMP_IBDMDB_2019['BMI_at_baseline'] = df_meta_iHMP_IBDMDB_2019['BMI_at_baseline'].fillna(0)

In [70]:
df_meta_iHMP_IBDMDB_2019

,Sample,Gender,BMI_at_baseline,Height_at_baseline,Weight_at_baseline,is_healthy
0,CSM5MCVN,0,0.0,0.0,0.0,0
1,CSM5MCWE,1,0.0,0.0,0.0,0
2,CSM5MCX3,1,20.1,180.0,65.0,0
3,CSM5MCXL,0,0.0,0.0,0.0,0
4,CSM5MCY8,0,30.9,165.0,84.0,0
...,...,...,...,...,...,...
377,PSMA267P,1,0.0,0.0,0.0,0
378,PSMA269W,0,0.0,0.0,0.0,0
379,PSMA26A3,0,0.0,0.0,0.0,0
380,PSMB4MBK,1,0.0,0.0,0.0,0


In [71]:
combined_df_inference_iHMP_IBDMDB_2019 = pd.merge(df_meta_iHMP_IBDMDB_2019, df_inference_iHMP_IBDMDB_2019, on='Sample', how='left')

In [72]:
combined_df_inference_iHMP_IBDMDB_2019

,Sample,Gender,BMI_at_baseline,Height_at_baseline,Weight_at_baseline,is_healthy,average_rho
0,CSM5MCVN,0,0.0,0.0,0.0,0,0.933238
1,CSM5MCWE,1,0.0,0.0,0.0,0,0.933323
2,CSM5MCX3,1,20.1,180.0,65.0,0,0.932071
3,CSM5MCXL,0,0.0,0.0,0.0,0,0.932730
4,CSM5MCY8,0,30.9,165.0,84.0,0,0.931674
...,...,...,...,...,...,...,...
377,PSMA267P,1,0.0,0.0,0.0,0,0.932335
378,PSMA269W,0,0.0,0.0,0.0,0,0.931577
379,PSMA26A3,0,0.0,0.0,0.0,0,0.933264
380,PSMB4MBK,1,0.0,0.0,0.0,0,0.931577


In [74]:
combined_df_inference_and_genera_iHMP_IBDMDB_2019 = pd.merge(combined_df_inference_iHMP_IBDMDB_2019, df_genera_iHMP_IBDMDB_2019, on='Sample', how='left')

In [75]:
combined_df_inference_and_genera_iHMP_IBDMDB_2019.set_index('Sample', inplace=True)	

In [76]:
combined_df_inference_and_genera_iHMP_IBDMDB_2019

,Gender,BMI_at_baseline,Height_at_baseline,Weight_at_baseline,is_healthy,average_rho,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Vicingaceae;g__BCD18,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__CAG-272;g__Flemingibacterium,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__F0428,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Peptostreptococcales;f__Acidaminobacteraceae;g__Fusibacter_A,...,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactobacillales;f__Lactobacillaceae;g__Paralactobacillus,d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Acetobacterales;f__Acetobacteraceae;g__JAFKFI01,d__Bacteria;p__Bacteroidota;c__Chlorobia;o__Chlorobiales;f__Chloroherpetonaceae;g__GBChlB,d__Bacteria;p__Actinobacteriota;c__Actinomycetia;o__Propionibacteriales;f__Actinopolymorphaceae;g__Tenggerimyces,d__Bacteria;p__Patescibacteria;c__ABY1;o__UBA1558;f__GWA2-36-10;g__JABHSK01,d__Bacteria;p__Firmicutes;c__Bacilli;o__Tepidibacillales;f__Tepidibacillaceae;g__Vulcanibacillus,d__Bacteria;p__Acidobacteriota;c__UBA6911;o__UBA6911;f__UBA6911;g__JAAYAM01,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactobacillales;f__Aerococcaceae;g__Ignavigranum,d__Bacteria;p__Patescibacteria;c__Saccharimonadia;o__Saccharimonadales;f__UBA4665;g__PMNU01,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactobacillales;f__Lactobacillaceae;g__Acetilactobacillus
Sample,,,,,,,,,,,,,,,,,,,,,
CSM5MCVN,0,0.0,0.0,0.0,0,0.933238,0.0,0.000000e+00,0.000000,0.000000e+00,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
CSM5MCWE,1,0.0,0.0,0.0,0,0.933323,0.0,4.103627e-06,0.000028,6.654530e-07,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
CSM5MCX3,1,20.1,180.0,65.0,0,0.932071,0.0,2.404909e-06,0.000008,0.000000e+00,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
CSM5MCXL,0,0.0,0.0,0.0,0,0.932730,0.0,6.116719e-07,0.000001,0.000000e+00,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
CSM5MCY8,0,30.9,165.0,84.0,0,0.931674,0.0,2.626476e-06,0.000007,4.736268e-07,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PSMA267P,1,0.0,0.0,0.0,0,0.932335,0.0,1.417163e-06,0.000000,0.000000e+00,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
PSMA269W,0,0.0,0.0,0.0,0,0.931577,0.0,1.967296e-06,0.000002,0.000000e+00,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
PSMA26A3,0,0.0,0.0,0.0,0,0.933264,0.0,1.304739e-06,0.000002,0.000000e+00,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0


In [77]:
X = combined_df_inference_and_genera_iHMP_IBDMDB_2019.drop('is_healthy', axis=1)
y = combined_df_inference_and_genera_iHMP_IBDMDB_2019['is_healthy']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [78]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [79]:
y_pred = rf.predict(X_test)

In [80]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8311688311688312


In [81]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print(importances)

average_rho                                                                                                                    0.044708
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Metaruminococcus                            0.007252
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__CAG-272;g__Flemingibacterium                                   0.005975
d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Ruthenibacterium                            0.004605
d__Bacteria;p__Cyanobacteria;c__Vampirovibrionia;o__Gastranaerophilales;f__Gastranaerophilaceae;g__Stercorousia                0.003233
                                                                                                                                 ...   
d__Bacteria;p__Verrucomicrobiota;c__Verrucomicrobiae;o__Methylacidiphilales;f__Methylacidiphilaceae;g__Methylacidimicrobium    0.000000
d__Bacteria;p__Proteobacteria;c__Gammaproteobact

In [82]:



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Initialize and train the Support Vector Classifier
# 'C' is the regularization parameter
model = SVC(kernel='linear', C=1.0)
model.fit(X_train, y_train)

# 5. Predict and Evaluate
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.77
              precision    recall  f1-score   support

           0       0.81      0.89      0.85        56
           1       0.60      0.43      0.50        21

    accuracy                           0.77        77
   macro avg       0.70      0.66      0.67        77
weighted avg       0.75      0.77      0.75        77

